# 02 - Statistical Analysis

This notebook performs statistically defensible analysis for Rossmann sales drivers.
Focus: inference quality, effect size, diagnostics, and time-aware validation.


## Planned Statistical Analyses (Appropriate for Rossmann)

1. Data conditioning and assumptions
- Restrict inference to open-store days with positive sales.
- Merge store-level covariates and create calendar controls.
- Use log-sales (`log1p(Sales)`) to stabilize variance.

2. Hypothesis tests for key business levers
- Promo impact: Welch t-test + Mann-Whitney U (robust to non-normality).
- Holiday effect: Kruskal-Wallis across `StateHoliday` groups.
- Store-type and assortment differences: one-way ANOVA on log-sales (plus robust alternatives).

3. Effect size reporting (not only p-values)
- Cohen's d / Cliff's delta for binary contrasts.
- Group mean deltas in original sales units for interpretability.

4. Regression inference with robust uncertainty
- OLS on log-sales with calendar/store covariates and `HC3` robust SE.
- Diagnose heteroskedasticity (Breusch-Pagan), residual normality (Jarque-Bera), autocorrelation (Durbin-Watson).
- Check multicollinearity (VIF).

5. Hierarchical structure handling
- Mixed-effects model with random intercept by `Store` on a sampled subset.
- Compare with pooled OLS to quantify store-level heterogeneity.

6. Quantile regression
- Estimate Promo/holiday effects at lower/median/upper sales quantiles.
- Useful because business interventions can behave differently for low vs high performing stores.

7. Time-aware validation for statistical models
- Train/validation split by date (not random split).
- Report RMSPE and RMSE on held-out time window.

These analyses are appropriate because Rossmann is panel data (stores over time), non-normal in raw scale, and sensitive to calendar/promo effects with likely heteroskedastic residuals.


## How To Use This Notebook
- Run top-to-bottom; each section builds on prior conditioning.
- Interpret model outputs using both significance and effect size.
- Prefer business-meaningful metrics (uplift, RMSPE) over p-values alone.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)


In [ ]:
RAW_DIR = Path("../data/raw")
FIG_DIR = Path("../reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(RAW_DIR / "train.csv", parse_dates=["Date"])
store = pd.read_csv(RAW_DIR / "store.csv")

df = train.merge(store, on="Store", how="left")
print(df.shape)
df.head()


In [ ]:
# Conditioning for valid sales inference

df = df[(df["Open"] == 1) & (df["Sales"] > 0)].copy()

# Minimal feature engineering
for c in ["CompetitionDistance", "CompetitionOpenSinceMonth", "CompetitionOpenSinceYear", "Promo2SinceWeek", "Promo2SinceYear"]:
    if c in df.columns:
        df[c] = df[c].fillna(0)

for c in ["StoreType", "Assortment", "PromoInterval", "StateHoliday"]:
    if c in df.columns:
        df[c] = df[c].fillna("Unknown")

df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
df["log_sales"] = np.log1p(df["Sales"])

print(df.shape)


In [ ]:
# Quick distribution check
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["Sales"], bins=80, kde=True, ax=ax[0])
ax[0].set_title("Sales")
sns.histplot(df["log_sales"], bins=80, kde=True, ax=ax[1], color="tab:orange")
ax[1].set_title("log1p(Sales)")
plt.tight_layout()
plt.savefig(FIG_DIR / "stats_sales_vs_logsales.png", dpi=140)
plt.show()


### Interpretation Guidance: Promo Tests
- If p-values are small and effect size is material, promo likely shifts sales meaningfully.
- Check whether uplift magnitude is operationally significant, not only statistically significant.


## 1) Promo Effect: Hypothesis Tests + Effect Size

- Null hypothesis: mean/median sales are equal for `Promo=1` and `Promo=0`.
- We report both Welch t-test and Mann-Whitney U.


In [ ]:
# Compare promo vs non-promo distributions with both parametric and non-parametric tests
promo_1 = df.loc[df["Promo"] == 1, "Sales"]
promo_0 = df.loc[df["Promo"] == 0, "Sales"]

welch = stats.ttest_ind(promo_1, promo_0, equal_var=False)
mannw = stats.mannwhitneyu(promo_1, promo_0, alternative="two-sided")

def cohens_d(x, y):
    x = np.asarray(x)
    y = np.asarray(y)
    nx, ny = len(x), len(y)
    sx, sy = x.std(ddof=1), y.std(ddof=1)
    pooled = np.sqrt(((nx - 1) * sx**2 + (ny - 1) * sy**2) / (nx + ny - 2))
    return (x.mean() - y.mean()) / pooled

print("Welch t-test:", welch)
print("Mann-Whitney U:", mannw)
print("Mean sales (Promo=1):", round(promo_1.mean(), 2))
print("Mean sales (Promo=0):", round(promo_0.mean(), 2))
print("Mean delta:", round(promo_1.mean() - promo_0.mean(), 2))
print("Cohen's d:", round(cohens_d(promo_1, promo_0), 3))


## 2) Holiday and Store Structure Tests


In [ ]:
# Multi-group tests detect structural differences across holiday/store categories
# Kruskal-Wallis for StateHoliday groups
holiday_groups = [g["Sales"].values for _, g in df.groupby(df["StateHoliday"].astype(str)) if len(g) > 30]
kw = stats.kruskal(*holiday_groups)
print("Kruskal-Wallis StateHoliday:", kw)

# ANOVA on log-sales for StoreType and Assortment
anova_storetype = smf.ols("log_sales ~ C(StoreType)", data=df).fit()
anova_assort = smf.ols("log_sales ~ C(Assortment)", data=df).fit()

print("StoreType ANOVA p-value:", sm.stats.anova_lm(anova_storetype, typ=2).iloc[0, 3])
print("Assortment ANOVA p-value:", sm.stats.anova_lm(anova_assort, typ=2).iloc[0, 3])


### Why Robust OLS Here
- Rossmann sales are heteroskedastic in raw scale.
- Log transform + HC3 robust SE provides more stable coefficient inference.


## 3) Robust OLS on log-sales

Model includes operational, competitive, and calendar controls.
HC3 robust standard errors reduce heteroskedasticity sensitivity.


In [ ]:
# Fit interpretable baseline model with operational and calendar controls
formula = (
    "log_sales ~ Promo + SchoolHoliday + C(StateHoliday) + C(StoreType) + C(Assortment) + "
    "CompetitionDistance + Promo2 + C(DayOfWeek) + C(Month) + Year"
)

ols = smf.ols(formula, data=df).fit(cov_type="HC3")
print(ols.summary().tables[1])


In [ ]:
# Diagnose key OLS assumptions; violations inform robust/hierarchical alternatives
# Residual diagnostics
resid = ols.resid
exog = ols.model.exog

bp_lm, bp_lm_p, bp_f, bp_f_p = het_breuschpagan(resid, exog)
dw = durbin_watson(resid)
jb_stat, jb_p, _, _ = jarque_bera(resid)

print(f"Breusch-Pagan p-value: {bp_lm_p:.6g}")
print(f"Durbin-Watson: {dw:.4f}")
print(f"Jarque-Bera p-value: {jb_p:.6g}")


In [ ]:
# Multicollinearity check (sampled for speed)
num_cols = [
    "Promo", "SchoolHoliday", "CompetitionDistance", "Promo2",
    "DayOfWeek", "Month", "Year", "Customers"
]

vif_df = df[num_cols].dropna().sample(min(120000, df[num_cols].dropna().shape[0]), random_state=42)
X = sm.add_constant(vif_df)

vif = pd.DataFrame({
    "feature": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})

vif.sort_values("VIF", ascending=False)


### Why Mixed Effects
- Observations are grouped by store; pooled OLS can understate uncertainty.
- Random intercepts absorb persistent store-level baseline differences.


## 4) Mixed-Effects Model (Store Random Intercept)

Captures persistent store-level differences while estimating global fixed effects.
Fitted on a subset for practical runtime.


In [ ]:
# Subsample for runtime while preserving cross-store variation for mixed model fitting
mix_cols = ["log_sales", "Promo", "SchoolHoliday", "CompetitionDistance", "Promo2", "Store", "DayOfWeek", "Month"]
mix_df = df[mix_cols].dropna().copy()
mix_df = mix_df.sample(min(200000, len(mix_df)), random_state=42)

mixed = smf.mixedlm(
    "log_sales ~ Promo + SchoolHoliday + CompetitionDistance + Promo2 + C(DayOfWeek) + C(Month)",
    data=mix_df,
    groups=mix_df["Store"]
).fit(reml=False, method="lbfgs", maxiter=200)

print(mixed.summary())


## 5) Quantile Regression

Estimates whether predictors have different effects in low- vs high-sales regimes.


In [ ]:
# Quantile regression reveals whether effects differ at low/median/high sales levels
q_formula = "log_sales ~ Promo + SchoolHoliday + CompetitionDistance + C(DayOfWeek) + C(Month)"
qs = [0.1, 0.5, 0.9]
qr_rows = []
for q in qs:
    qr = smf.quantreg(q_formula, df).fit(q=q)
    qr_rows.append({"quantile": q, "promo_coef": qr.params.get("Promo", np.nan), "schoolholiday_coef": qr.params.get("SchoolHoliday", np.nan)})

pd.DataFrame(qr_rows)


### Validation Principle
- Forecasting problems require chronological holdout.
- Random split can overestimate performance due to temporal leakage.


## 6) Time-Aware Validation (RMSPE, RMSE)

We avoid random split and hold out the latest period.


In [ ]:
# Evaluate in time order to emulate real forecasting deployment
def rmspe(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = y_true != 0
    return np.sqrt(np.mean(np.square((y_true[mask] - y_pred[mask]) / y_true[mask])))

cutoff = df["Date"].quantile(0.85)
train_df = df[df["Date"] <= cutoff].copy()
valid_df = df[df["Date"] > cutoff].copy()

val_model = smf.ols(formula, data=train_df).fit(cov_type="HC3")
valid_pred_log = val_model.predict(valid_df)
valid_pred_sales = np.expm1(valid_pred_log)

rmse = np.sqrt(np.mean((valid_df["Sales"] - valid_pred_sales) ** 2))
r = rmspe(valid_df["Sales"].values, valid_pred_sales.values)

print("Cutoff:", cutoff)
print("Validation RMSE:", round(rmse, 2))
print("Validation RMSPE:", round(r, 4))


## Interpretation Checklist

- Promote decisions should be based on effect size + uplift confidence, not p-values only.
- If residual diagnostics remain poor, prefer robust/quantile/tree-based models for prediction.
- Mixed-effects significance supports hierarchical modeling by store.
- Use this notebook's inference to guide feature engineering in notebook 03.
